### RAG Pipeline - Data Ingestion to vector DB Pileline

In [1]:
## importing necessary library

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [2]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all pdf files in the given directory"""
    pdf_dir=Path(pdf_directory)
    all_documents=[]
    
    # Find all pdf files in the directory
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files in the directory.")
    
    for pdf_file in pdf_files:
        print(f"\nprocessing file: {pdf_file.name}")
        #Load the pdf file
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()
            
            # Add source info to the metadata
            for doc in documents:
                doc.metadata['source']=pdf_file.name
                doc.metadata['file_type']="pdf"
                
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} documents from {pdf_file.name}")
            
        except Exception as e:
            print(f"Error loading {pdf_file.name}: {e}")
            
    print(f"\nTotal documents loaded: {len(all_documents)}")
    
    return all_documents

all_pdf_documents=process_all_pdfs("../data/pdfFiles")

In [3]:
all_pdf_documents

In [4]:
### Text Splitting and chunking the documents

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks.")
    
    # Show an Example of chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

### Embedding and Vectore Store DB


In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings (dimension: 384 for all-MiniLM-L6-v2)
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

### Vector Store

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore


In [ ]:
chunks

In [ ]:
### Convert the text to embeddings

## Extract the text content from the chunks
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings
print(f"Generating embeddings for {len(texts)} chunks...")
embeddings=embedding_manager.generate_embeddings(texts)

## store in the vector database
vectorstore.add_documents(chunks,embeddings)

### Retriever Pipeline From VectorStore

In [43]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [44]:
rag_retriever.retrieve("What is data")

Retrieving documents for query: 'What is data'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_70f0d535_0',
  'content': 'LEC-1: Introduction to DBMS \n1. What is Data?\na. Data is a collection of raw, unorganized facts and details like text, observations, figures, symbols,\nand descriptions of things etc.\nIn other words, data does not carry any specific purpose and has no significance by itself.\nMoreover, data is measured in terms of bits and bytes – which are basic units of information in the\ncontext of computer storage and processing.\nb. Data can be recorded and doesn’t have any meaning unless processed.\n2. Types of Data\na. Quantitative\ni. Numerical form\nii. Weight, volume, cost of an item.\nb. Qualitative\ni. Descriptive, but not numerical.\nii. Name, gender, hair color of a person.\n3. What is Information?\na. Info. Is processed, organized, and structured data.\nb. It provides context of the data and enables decision making.\nc. Processed data that make sense to us.\nd. Information is extracted from the data, by analyzing and interpreting pieces of data

### RAG Pipeline: VectorDB To LLM Output Generation

In [56]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.7, max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question
        Context:
        {context}
        Question: {query}
        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content



In [ ]:
# Question in COntext

answer=rag_simple("What is Thread Scheduling", rag_retriever, llm)
print("Answer:", answer)

Retrieving documents for query: 'What is Thread Scheduling'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.92it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: Thread Scheduling refers to the process by which threads are scheduled for execution based on their priority. The operating system assigns processor time slices to each thread, allowing them to execute within the runtime. This means that the threads are given a specific amount of time to run before the operating system switches to another thread, ensuring that all threads are executed in a fair and efficient manner.


In [ ]:
#Question outof context

answer=rag_simple("Who is Kapil Gupta",rag_retriever,llm)
print("Answer:", answer)

Retrieving documents for query: 'Who is Kapil Gupta'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.68it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found to answer the question.
